# 課程紀錄
此程式可以相對容易的從全人系統上把修課紀錄抓下來
需要安裝以下依賴
- requests
- pandas
- beautifulsoup4

In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import create_urllib3_context
import csv
import pandas as pd
from bs4 import BeautifulSoup
import io

course_information = set()
with open('course_information.csv', mode='r', newline='') as file:
    reader = csv.DictReader(file)
    for row in reader:
        course_information.add(row["course_id"])

In [12]:
class LegacyVersionAdapter(HTTPAdapter):
    def init_poolmanager(self, *args, **kwargs):
        context = create_urllib3_context()
        # 💡 核心：將 SSL 的 Options 加上允許不安全重新協商的旗標
        # 0x40000 代表 SSL_OP_LEGACY_SERVER_CONNECT
        context.options |= 0x40000  
        kwargs["ssl_context"] = context
        return super().init_poolmanager(*args, **kwargs)

session = requests.Session()
session.mount("https://", LegacyVersionAdapter())

## 使用步驟
1. 登入政大全人系統
2. 按F12打開瀏覽器調適介面，找到cookies的存放位置
3. 將cookies中名字為.LDAPAUTH, JSESSIONID的值複製設為下方變數值

In [14]:
ID = "113703014"
JSESSIONID = "57FFA31E29F359E22A7789BC2E557CEB"
LDAPAUTH = "43FB96065713ECB319FF98F65827B4D3534565C700117B928142E93872E6371278B5B15A93AA861C6FEE663F57C4A8AD9A5D1CA1F56810883079254EB1751E47CCC11F85E2122FB097D281A031FE92CDCAE9F24DF2A841F813ABC7CC9D500CB24FC5D729DADB68E41D68D681D77D2ADD356916AE2565A59908DA5DD262D0DFF0DA9CA594"

def fetch_personal_information(): 
    headers = {
        "Host": "moltke.nccu.edu.tw",
        "Referer": "https://moltke.nccu.edu.tw/selfDevelopV2025_SSO/Home",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36"
    }
    cookies = {
        "JSESSIONID":  JSESSIONID,
        ".LDAPAUTH": LDAPAUTH
    }

    try:
        response = session.get("https://moltke.nccu.edu.tw/selfDevelopV2025_SSO/AL", headers=headers, cookies=cookies, timeout=15)
        response.raise_for_status()
        return response
    except Exception as e:
        print(f"Error: {e}")
        return None

response = fetch_personal_information()


4. 可以在course_record.csv中的到結果，有些0學分的課程狀態可能會被標成unknown，要手動改成passed
5. 把course_record.csv中的值加到seeds/course_record.csv中，又多了一人的資料，好ㄟ

In [ ]:
if response is not None and response.status_code == 200:
    soup = BeautifulSoup(response.text, "lxml")
    target_div = soup.find("div", id="item-9")
    dfs = pd.read_html(
        io.StringIO(str(target_div)),
        converters={
            "科目代號": str
        }
    )
    
    course_records = [['student_id','course_id','semester','grade','status']]
    for df in dfs:
        if "科目代號" in df.columns:
            for index, row in df.iterrows():
                if row["科目代號"] in course_information:
                    grade = 0
                    status = ""
                    try:
                        if float(row["成績"]) >= 60:
                            status = "passed"
                        else:
                            status = "failed"
                        grade = row["成績"]
                    except:
                        match row["成績"]:
                            case "成績未到或無成績":
                                status = "unknown"
                            case "通過":
                                status = "passed"
                        grade = ""
                        
                    course_records.append([
                        ID,
                        row["科目代號"],
                        f"{row["學年"]}-{row["學期"]}",
                        grade,
                        status
                    ])
    
    with open('course_record.csv', 'w', encoding='utf-8', newline="") as f:
        writer = csv.writer(f)
        writer.writerows(course_records)